# Datathon 2026 — Faz 0: Teşhis (Diagnostic)

**Amaç:** CONTEXT.md §8'deki bilinmeyenleri tek çalıştırmada teyit etmek ve §9 teşhis kodunu genişletmek.

Bu notebook hiçbir model eğitmez — sadece veriyi **tanır**. Sonunda Faz 0'ın asıl teslimatı olan
**sabit-ortalama submission**'ı üretir (katılımı kilitler + varyans tabanını verir).

Teyit edilecekler:
1. `/kaggle/input/` klasör adı
2. satır/kolon sayısı + dtype dağılımı
3. `mentor_feedback_text` dili (TR/EN/karışık)
4. hedef (`career_success_score`) dağılımı
5. `sample_submission.csv` birebir yapısı
6. eksik değerler (özellikle metinde)

## 1. Input klasörünü bul ve dosyaları yükle

In [ ]:
import pandas as pd, numpy as np, glob, os, re

# Klasör adını otomatik bul (CONTEXT §8: klasör adı bilinmiyor)
candidates = glob.glob('/kaggle/input/*')
print("input altındaki klasörler:", candidates)
base = candidates[0]
print("Seçilen klasör:", base)
print("İçerik:", os.listdir(base))

train = pd.read_csv(f'{base}/train.csv')
test  = pd.read_csv(f'{base}/test_x.csv')   # DİKKAT: test.csv değil, test_x.csv
sub   = pd.read_csv(f'{base}/sample_submission.csv')

print("\ntrain :", train.shape)
print("test_x:", test.shape)
print("sample_submission:", sub.shape)

## 2. sample_submission yapısı (§8.5)
Submission'ın birebir kolon adlarını ve ID hizasını doğrula.

In [ ]:
print("sample_submission kolonları:", list(sub.columns))
print(sub.head())
print()
# ID kolonu test ile hizalı mı? (satır sayısı ve değer kümesi)
id_col = sub.columns[0]
print("ID kolonu:", id_col)
print("sub satır =", len(sub), " | test satır =", len(test))
if id_col in test.columns:
    print("test ID == sub ID (aynı küme & sıra)?",
          test[id_col].reset_index(drop=True).equals(sub[id_col].reset_index(drop=True)))
print("hedef kolon adı submission'da:", [c for c in sub.columns if c != id_col])

## 3. Hedef dağılımı — `career_success_score` (§8.4)
MSE metriği uçlardaki hatayı domine eder; dağılımın şeklini ve sabit-ortalama tabanını gör.

In [ ]:
y = train['career_success_score']
print(y.describe())
print("\nskewness (çarpıklık):", round(y.skew(), 4))
print("kurtosis (basıklık) :", round(y.kurt(), 4))
print("min / max           :", y.min(), "/", y.max())
print("0-100 dışında değer var mı? <0:", int((y < 0).sum()), " >100:", int((y > 100).sum()))

# Sabit-ortalama tahmininin MSE'si = hedefin varyansı (popülasyon).
# Bu, "hiçbir şey öğrenmeden" elde edilecek MSE -> LB'deki 86 ile kıyas için taban.
print("\n--- Faz 0 teşhis tabanı ---")
print("mean      :", round(y.mean(), 6))
print("var (pop) :", round(y.var(ddof=0), 6), " <- sabit-ortalama submission'ın beklenen ~public MSE'si")
print("std       :", round(y.std(ddof=0), 6))

## 4. `mentor_feedback_text` dili (§8.3)
İşin ayrışma noktası. Embedding modeli seçimi dile bağlı — TR mı EN mi karışık mı?

In [ ]:
txt = train['mentor_feedback_text']
print("metin eksik (NaN) sayısı:", int(txt.isna().sum()), "/", len(txt))
print("boş string sayısı       :", int((txt.fillna('').str.strip() == '').sum()))
lengths = txt.fillna('').str.len()
print("metin uzunluğu (char) describe:")
print(lengths.describe())

print("\n--- İlk 5 örnek ---")
for i, t in enumerate(txt.dropna().head(5).tolist()):
    print(f"[{i}] {t}")

# Basit TR/EN sezgisi: Türkçe'ye özgü karakterler + sık fonksiyon kelimeleri
sample = txt.dropna().head(2000).str.lower()
tr_chars = sample.str.contains(r'[çğışöü]', regex=True).mean()
tr_words = sample.str.contains(r'\b(ve|bir|ile|için|çok|daha|olarak|ancak|gelişim)\b', regex=True).mean()
en_words = sample.str.contains(r'\b(the|and|with|for|very|more|however|development|student)\b', regex=True).mean()
print("\n--- Dil sezgisi (örneklem=2000) ---")
print("TR'ye özgü karakter içeren oran :", round(float(tr_chars), 3))
print("TR fonksiyon kelimesi oranı     :", round(float(tr_words), 3))
print("EN fonksiyon kelimesi oranı     :", round(float(en_words), 3))
print("=> Tahmin:", "TR ağırlıklı" if tr_chars > 0.3 or tr_words > en_words else "EN ağırlıklı / karışık (manuel doğrula)")

## 5. Eksik değerler (§8.6)
Tüm kolonlarda NaN durumu; özellikle metin alanına dikkat.

In [ ]:
na = train.isna().sum()
na = na[na > 0].sort_values(ascending=False)
if len(na):
    print("Eksik değer olan kolonlar (train):")
    print((na.to_frame('na_count').assign(na_pct=lambda d: (d['na_count']/len(train)*100).round(2))))
else:
    print("train'de eksik değer YOK.")

print("\n--- test_x eksik değerleri ---")
na_t = test.isna().sum(); na_t = na_t[na_t > 0].sort_values(ascending=False)
print(na_t if len(na_t) else "test_x'de eksik değer YOK.")

## 6. Kolon envanteri & dtype'lar (§8.2)
Gerçek kolon sayısı, dtype dağılımı, kategorik vs sayısal ayrımı (CatBoost için).

In [ ]:
print("dtype dağılımı:")
print(train.dtypes.value_counts())

ID = 'student_id'
TARGET = 'career_success_score'
TEXT = 'mentor_feedback_text'

cat_cols = [c for c in train.columns if train[c].dtype == 'object' and c not in (ID, TEXT)]
num_cols = [c for c in train.columns if c not in (ID, TARGET, TEXT) and c not in cat_cols]

print("\nID     :", ID)
print("TARGET :", TARGET)
print("TEXT   :", TEXT)
print("\nKategorik kolonlar (%d):" % len(cat_cols), cat_cols)
print("\nSayısal kolonlar (%d):" % len(num_cols), num_cols)

# train'de olup test'te olmayan / tersi (hedef hariç leak kontrolü)
only_train = set(train.columns) - set(test.columns)
only_test  = set(test.columns) - set(train.columns)
print("\nSadece train'de:", only_train, "(career_success_score beklenir)")
print("Sadece test'te :", only_test, "(boş olmalı)")

# Kategoriklerin kardinalitesi
print("\nKategorik kardinalite:")
for c in cat_cols:
    print(f"  {c}: {train[c].nunique()} benzersiz | örnek: {train[c].dropna().unique()[:5].tolist()}")

## 7. Faz 0 teslimatı — sabit-ortalama submission

İlk submission'ı buradan üret. Bu, katılımı kilitler ve LB'deki ~86 ile kıyaslanacak varyans tabanını verir.
Tahminler `np.clip(0, 100)` ile sıkıştırılır (CONTEXT zorunlu adım) — sabit ortalama zaten aralıkta ama kural gereği uygulanır.

In [ ]:
mean_pred = float(train['career_success_score'].mean())
id_col = sub.columns[0]
target_col = [c for c in sub.columns if c != id_col][0]

submission = sub.copy()
submission[id_col] = test[id_col].values if id_col in test.columns else sub[id_col].values
submission[target_col] = np.clip(mean_pred, 0, 100)

out = '/kaggle/working/sub_phase0_constant_mean.csv'
submission.to_csv(out, index=False)
print("Yazıldı:", out)
print("Sabit tahmin değeri:", round(mean_pred, 6))
print(submission.head())
print("Beklenen public MSE ~ hedef varyansı:", round(train['career_success_score'].var(ddof=0), 4))

## Sonraki adım — Faz 1 (Baseline)

Yukarıdaki çıktılar §8'i kapatınca: binned-stratified 5-fold CV + CatBoost (tüm sayısal + kategorik), OOF MSE logla.
§8 sonuçlarını CONTEXT.md'ye işleyip Faz 1 pipeline'ına geçeceğiz.